# Dynamo frontend smoke test

This notebook calls the OpenAI-compatible Dynamo HTTP frontend. Start the frontend on port 8000 (for example `python -m dynamo.frontend --http-port 8000` with workers registered), then run the cells in order.

Endpoints covered: `GET /health`, `GET /v1/models`, `POST /v1/chat/completions`.

In [1]:
import json
import urllib.error
import urllib.request

BASE_URL = "http://localhost:8000"
TIMEOUT_SEC = 120


from typing import Any, Dict, Optional, Tuple


def _request(method: str, path: str, body: Optional[Dict[str, Any]] = None) -> Tuple[int, bytes]:
    url = BASE_URL.rstrip("/") + path
    data = json.dumps(body).encode("utf-8") if body is not None else None
    req = urllib.request.Request(
        url,
        data=data,
        method=method,
        headers={"Content-Type": "application/json"} if body is not None else {},
    )
    with urllib.request.urlopen(req, timeout=TIMEOUT_SEC) as resp:
        return resp.status, resp.read()


def get_json(path: str) -> Tuple[int, dict]:
    status, raw = _request("GET", path, None)
    return status, json.loads(raw.decode("utf-8"))


def post_json(path: str, payload: dict) -> Tuple[int, dict]:
    status, raw = _request("POST", path, payload)
    return status, json.loads(raw.decode("utf-8"))


print(f"Target: {BASE_URL}")

Target: http://localhost:8000


In [2]:
try:
    status, health = get_json("/health")
    print(f"GET /health -> {status}")
    print(json.dumps(health, indent=2))
except urllib.error.HTTPError as e:
    print(f"GET /health -> HTTP {e.code}")
    print(e.read().decode("utf-8", errors="replace"))
except urllib.error.URLError as e:
    print("Connection failed — is the frontend listening on", BASE_URL, "?")
    raise

GET /health -> 200
{
  "status": "healthy",
  "endpoints": [
    "dyn://dynamo-system-nemotron-super-fp8-sglang-disagg-ff505fcf.backend.generate",
    "dyn://dynamo-system-nemotron-super-fp8-sglang-disagg-ff505fcf.prefill.generate"
  ],
  "instances": [
    {
      "component": "backend",
      "endpoint": "generate",
      "namespace": "dynamo-system-nemotron-super-fp8-sglang-disagg-ff505fcf",
      "instance_id": 6219441052810981,
      "transport": {
        "tcp": "10.244.4.119:45493/16188c76a77ee5/generate"
      }
    },
    {
      "component": "prefill",
      "endpoint": "generate",
      "namespace": "dynamo-system-nemotron-super-fp8-sglang-disagg-ff505fcf",
      "instance_id": 895673035031716,
      "transport": {
        "tcp": "10.244.5.179:46559/32e9c211684a4/generate"
      }
    }
  ]
}


In [3]:
status, models_doc = get_json("/v1/models")
print(f"GET /v1/models -> {status}")
print(json.dumps(models_doc, indent=2)[:4000])

model_ids = [m["id"] for m in models_doc.get("data", []) if "id" in m]
if not model_ids:
    raise RuntimeError("No models in /v1/models — register a backend before chat tests.")
MODEL = model_ids[0]
print(f"\nUsing model: {MODEL!r}")

GET /v1/models -> 200
{
  "object": "list",
  "data": [
    {
      "id": "nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-FP8",
      "object": "model",
      "created": 1776455947,
      "owned_by": "nvidia"
    }
  ]
}

Using model: 'nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-FP8'


In [5]:
chat_payload = {
    "model": MODEL,
    "messages": [
        {"role": "system", "content": "You are a helpful assistant. Reply in one short sentence."},
        {"role": "user", "content": "Say hello and confirm the Dynamo endpoint is working."},
    ],
    "max_tokens": 128,
    "temperature": 0.2,
    "stream": False,
}

status, completion = post_json("/v1/chat/completions", chat_payload)
print(f"POST /v1/chat/completions -> {status}")
print(json.dumps(completion, indent=2))

choice0 = completion["choices"][0]
msg = choice0.get("message", {})
print("\nAssistant:", msg.get("content", choice0))

POST /v1/chat/completions -> 200
{
  "id": "chatcmpl-e3b025e2-b726-4cd7-b269-2925d224a589",
  "choices": [
    {
      "index": 0,
      "message": {
        "content": "\n\nHello! The Dynamo endpoint is up and running.",
        "role": "assistant",
        "reasoning_content": "We need to respond in one short sentence. Say hello and confirm Dynamo endpoint is working. So something like: \"Hello! The Dynamo endpoint is up and running.\" That's one sentence. Ensure it's short.\n\n"
      },
      "finish_reason": "stop"
    }
  ],
  "created": 1776456300,
  "model": "nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-FP8",
  "object": "chat.completion",
  "usage": {
    "prompt_tokens": 38,
    "completion_tokens": 56,
    "total_tokens": 94
  },
  "nvext": {
    "timing": {
      "request_received_ms": 1776456300960,
      "total_time_ms": 1238.455481
    }
  }
}

Assistant: 

Hello! The Dynamo endpoint is up and running.


### Optional: OpenAI client

If you have `openai` installed, you can point the client at the same base URL (no API key required for typical local setups).

In [ ]:
try:
    from openai import OpenAI
except ImportError:
    print("Skip: pip install openai")
else:
    client = OpenAI(base_url=BASE_URL.rstrip("/") + "/v1", api_key="not-needed")
    r = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": "Reply with the single word: ok"}],
        max_tokens=16,
    )
    print(r.choices[0].message.content)